## **Performing it manually**
### This section implements a Naïve Bayes model from scratch using the provided dataset

In [ ]:
import math
import re
from collections import Counter

data = [
    ("Free money now!!!", "SPAM"),
    ("Hi mom, how are you?", "HAM"),
    ("Lowest price for your meds", "SPAM"),
    ("Are we still on for dinner?", "HAM"),
    ("Win a free iPhone today", "SPAM"),
    ("Let's catch up tomorrow at the office", "HAM"),
    ("Meeting at 3 PM tomorrow", "HAM"),
    ("Get 50% off, limited time!", "SPAM"),
    ("Team meeting in the office", "HAM"),
    ("Click here for prizes!", "SPAM"),
    ("Can you send the report?", "HAM")
]

def tokenize(text):
    return re.findall(r'\b\w+\b', text.lower())

### a. Generate a Bag of Words (for word frequency) 
class_docs = {"SPAM": [d[0] for d in data if d[1] == "SPAM"],
              "HAM": [d[0] for d in data if d[1] == "HAM"]}

word_counts = {"SPAM": Counter(), "HAM": Counter()}
vocabulary = set()

for label, docs in class_docs.items():
    for doc in docs:
        tokens = tokenize(doc)
        word_counts[label].update(tokens)
        vocabulary.update(tokens)

print(f"Vocabulary Size: {len(vocabulary)}")

### b. Calculate the prior for the class HAM and SPAM 
total_docs = len(data)
priors = {label: len(docs) / total_docs for label, docs in class_docs.items()}
print(f"Priors: {priors}")

### c. Calculate the likelihood of the tokens with respect to the class 
def get_likelihood(word, label):
    # Laplace Smoothing: (count + 1) / (total words in class + vocab size)
    word_occurrence = word_counts[label][word]
    total_words_in_class = sum(word_counts[label].values())
    return (word_occurrence + 1) / (total_words_in_class + len(vocabulary))

### d. Determine the class of the following test sentences 
def predict_manual(sentence):
    tokens = tokenize(sentence)
    results = {}
    for label in ["SPAM", "HAM"]:
        log_prob = math.log(priors[label])
        for token in tokens:
            if token in vocabulary:
                log_prob += math.log(get_likelihood(token, label))
        results[label] = log_prob
    return max(results, key=results.get)

# i. Limited offer, click here! 
# ii. Meeting at 2 PM with the manager. 
test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

print("\n--- Manual Predictions ---")
for ts in test_sentences:
    print(f"Sentence: '{ts}' -> Prediction: {predict_manual(ts)}")

Vocabulary Size: 45
Priors: {'SPAM': 0.45454545454545453, 'HAM': 0.5454545454545454}

--- Manual Predictions ---
Sentence: 'Limited offer, click here!' -> Prediction: SPAM
Sentence: 'Meeting at 2 PM with the manager.' -> Prediction: HAM


## **Using Scikit-Learn**
### This section uses the scikit-learn package to achieve the same classification goals.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# Prepare training data
X_train = [d[0] for d in data]
y_train = [d[1] for d in data]

# Vectorize the text data
vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(X_train)

# Train the Multinomial Naïve Bayes classifier
clf = MultinomialNB()
clf.fit(X_vec, y_train)

### a. Determine the class of the following test sentences 
# i. Limited offer, click here!
# ii. Meeting at 2 PM with the manager. 
test_sents_sk = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager."
]

X_test = vectorizer.transform(test_sents_sk)
predictions = clf.predict(X_test)

print("--- Scikit-Learn Predictions ---")
for sent, pred in zip(test_sents_sk, predictions):
    print(f"Sentence: '{sent}' -> Prediction: {pred}")

--- Scikit-Learn Predictions ---
Sentence: 'Limited offer, click here!' -> Prediction: SPAM
Sentence: 'Meeting at 2 PM with the manager.' -> Prediction: HAM
